# Facebook Judol Comment Scraper — Colab / Kaggle

Notebook ini menjalankan scraper komentar promosi judi online dari repo
[alvisdiy/ScrapingFacebookComments](https://github.com/alvisdiy/ScrapingFacebookComments)
di Google Colab atau Kaggle. Satu kali jalan = cari postingan → ambil komentar →
analisis detektor promo, berulang otomatis sampai putaran/waktu yang kamu tentukan.

**Yang kamu butuhkan sebelum mulai:**
1. Akun Facebook **throwaway / khusus** (JANGAN akun utama — scraping login melanggar aturan Facebook dan akun bisa dibatasi).
2. Salah satu dari dua ini untuk login (lihat sel login di bawah):
   - `auth_state.json` — file sesi dari login sekali di komputermu sendiri (**disarankan**), atau
   - `login.csv` — email + password akun throwaway, diisi di komputermu sendiri.
3. Di **Kaggle**: nyalakan Internet di Settings notebook (kanan atas) sebelum menjalankan sel setup.

**Jalankan selnya berurutan dari atas ke bawah.** Jangan loncat sel setup.


In [ ]:
# 1. SETUP — clone repo, install dependensi, install browser Playwright.
import os

REPO_URL = "https://github.com/alvisdiy/ScrapingFacebookComments.git"
REPO_DIR = "ScrapingFacebookComments"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
os.chdir(REPO_DIR)
!git pull --ff-only
!pip install -q -r requirements.txt
!playwright install chromium
!playwright install-deps chromium
print("Setup selesai. File di repo:", sorted(os.listdir("."))[:12])


## Pulihkan sesi + data dari Google Drive (Colab saja, opsional)

Kalau kamu pernah menyimpan sesi/data ke Drive dari sel terakhir notebook ini,
jalankan sel di bawah supaya sesi dan dataset lamamu kembali — lalu kamu bisa
langsung loncat ke sel **Jalankan**. Di Kaggle, lewati sel ini.


In [ ]:
# 2. (COLAB SAJA) Pulihkan sesi + data dari Drive, kalau pernah disimpan.
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    !cp -r "/content/drive/MyDrive/fb-judol/data" ./data 2>/dev/null || echo "belum ada data di Drive"
    !cp "/content/drive/MyDrive/fb-judol/auth_state.json" ./auth_state.json 2>/dev/null || echo "belum ada sesi di Drive"
    print("auth_state.json ada:", os.path.exists("auth_state.json"))
    print("data/raw_comments.jsonl ada:", os.path.exists("data/raw_comments.jsonl"))
except ImportError:
    print("Bukan Colab — lewati sel ini.")


## Login — pilih SALAH SATU opsi

**Opsi A (disarankan): upload `auth_state.json`.**
Cara bikinnya di komputermu sendiri (sekali saja): clone repo, `pip install -r requirements.txt`,
`python3 login_csv.py --headed` (isi `login.csv` dari `logins.csv.template` dulu),
lalu file `auth_state.json` yang terbentuk itu yang kamu upload ke sini.
Setelah itu hapus `login.csv` di komputermu.

**Opsi B: upload `login.csv`, login lewat form di sini.**
Bisa, tapi kalau Facebook menampilkan checkpoint / kode 2FA, proses di Colab/Kaggle
tidak bisa diselesaikan secara interaktif (tidak ada jendela browser) dan login akan
gagal setelah 300 detik. Kalau begitu, pakai Opsi A.

**JANGAN PERNAH** kirim `login.csv` atau `auth_state.json` ke siapa pun / ke chat —
keduanya setara akses akun. Jangan commit keduanya ke GitHub (sudah di .gitignore).


In [ ]:
# 3a. LOGIN OPSI A — upload auth_state.json dari komputermu.
import os
try:
    from google.colab import files
    uploaded = files.upload()  # pilih auth_state.json
    for name in uploaded:
        os.replace(name, "auth_state.json")
except ImportError:
    print("Bukan Colab — upload auth_state.json lewat panel Input Kaggle ke folder kerja ini, lalu jalankan sel ini lagi.")

print("auth_state.json ada:", os.path.exists("auth_state.json"))


In [ ]:
# 3b. LOGIN OPSI B — upload login.csv, login lewat form (headless).
# Kalau sel ini gagal karena checkpoint/2FA, pakai Opsi A di atas.
import os
try:
    from google.colab import files
    uploaded = files.upload()  # pilih login.csv
    for name in uploaded:
        os.replace(name, "login.csv")
except ImportError:
    print("Bukan Colab — upload login.csv lewat panel Input Kaggle, lalu jalankan sel ini lagi.")

!python3 login_csv.py --headless
!rm -f login.csv  # hapus file password, apa pun hasilnya
print("Sesi tersimpan (auth_state.json ada):", os.path.exists("auth_state.json"))


## Jalankan — mode otomatis

Satu putaran = cari postingan baru → ambil komentarnya → analisis detektor → laporan.
`MAX_POSTS_PER_ROUND` itu batas **per putaran, bukan batas total** — putaran berikutnya
mengambil postingan baru lagi. `ROUNDS = 2` di bawah itu cuma contoh biar tes pertamamu
pendek; naikkan kalau sudah yakin jalan.

Mode: `random` = hanya postingan biasa (istilah netral, tanpa kata kunci judol) —
bagus untuk menangkap promo yang menempel di postingan biasa.
`all` = kata kunci judol + feed grup + random. `keywords` = kata kunci judol + feed grup.


In [ ]:
# 4. JALANKAN — ubah parameter di sini, lalu jalankan sel ini.
MODE = "random"           # all | keywords | random
MAX_POSTS_PER_ROUND = 5   # postingan baru per putaran (bukan batas total)
ROUNDS = 2                # jumlah putaran (naikkan kalau sudah yakin)
SLEEP = 180               # jeda detik antar putaran
MINUTES = 0               # 0 = tanpa batas waktu; mis. 120 = berhenti setelah 2 jam

!python3 auto.py --mode {MODE} --max-posts-per-round {MAX_POSTS_PER_ROUND} --max-rounds {ROUNDS} --sleep {SLEEP} --minutes {MINUTES}


## Lihat hasil


In [ ]:
# 5. HASIL — hitung ulang detektor di semua komentar yang terkumpul.
import os, json

if os.path.exists("data/raw_comments.jsonl"):
    !python3 analyze_spam.py
    summary = json.load(open("data/spam_summary.json", encoding="utf-8"))
    print("\nPROMO TOTAL:", summary["promo_comments"])
    print("Label:", summary["labels"])
    print("Komentar unik:", summary["distinct_comments"])
    print("Top brand spam:", list(summary["spam_brand_tokens_by_posts"].items())[:10])
else:
    print("Belum ada data — jalankan dulu sel JALANKAN di atas.")
!ls -lh data/ 2>/dev/null || echo "folder data belum ada"


In [ ]:
# 6. UNDUH dataset promo (CSV) ke komputermu.
try:
    from google.colab import files
    files.download("data/promo_comments.csv")
except ImportError:
    print("Bukan Colab. Di Kaggle: file ada di data/promo_comments.csv — simpan sebagai Output/Dataset sebelum sesi berakhir.")
except FileNotFoundError:
    print("promo_comments.csv belum ada — jalankan dulu sel JALANKAN + HASIL.")


## Simpan ke Google Drive (Colab saja) — supaya tidak hilang saat runtime mati

Runtime Colab itu sementara: kalau terputus / habis masa aktif, semua file di sini
hilang, termasuk sesi login dan dataset. Simpan ke Drive dengan sel di bawah, dan
pulihkan dengan sel nomor 2 di sesi berikutnya. Di Kaggle, simpan folder `data`
sebagai Output/Dataset sebelum sesi berakhir (maks 9 jam per sesi).


In [ ]:
# 7. (COLAB SAJA) Simpan sesi + data ke Google Drive.
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    !mkdir -p "/content/drive/MyDrive/fb-judol"
    !cp -r data "/content/drive/MyDrive/fb-judol/data" 2>/dev/null || echo "folder data belum ada"
    !cp auth_state.json "/content/drive/MyDrive/fb-judol/" 2>/dev/null || echo "auth_state.json tidak ada (belum login?)"
    print("Tersimpan di Drive: MyDrive/fb-judol/")
except ImportError:
    print("Bukan Colab — lewati sel ini. Di Kaggle, simpan folder data sebagai Output/Dataset.")


## Batasan yang jujur

- **Runtime sementara.** Colab gratis: idle ~90 menit, sesi maks ~12 jam. Kaggle: maks 9 jam
  per sesi, dan Internet harus dinyalakan manual. Jadi jangan harap jalan berhari-hari
  tanpa henti — jalankan beberapa putaran, simpan ke Drive/Output, lanjutkan lain kali
  (postingan yang sudah selesai tidak diulang).
- **Facebook bisa minta checkpoint** saat login dari IP datacenter Google/Kaggle. Kalau
  Opsi B gagal, pakai Opsi A (sesi dari komputermu sendiri).
- **Akun throwaway saja.** Jangan akun utama. Jangan bagikan `login.csv` /
  `auth_state.json` ke siapa pun.
- **Dataset promo = spam + candidate** (komentar promosi, apa pun labelnya); keluhan /
  peringatan tanpa unsur promo tidak masuk dataset.
